# SPECTRE-GRID | NOMS 2027 | Google Colab Pro (Tesla T4 15 GB)

Treinamento científico da arquitetura `CNN-1D + LSTM + GATConv` em GPU, utilizando snapshots temporais discretos $G_t = (V_t, E_t)$, treino/validação/teste disjuntos (60% / 20% / 20%) e pré-processamento ajustado apenas no treino.

**Não é um notebook de YOLOv8.** O notebook da Roboflow serviu somente como referência da interface do Colab.

**Protocolo e Capacidades Implementadas (`fix/leakage-free-pipeline`):**
- **Retomada Exata (`--resume` / `--save_every 1`):** `train.py` salva periodicamente no Google Drive o estado completo (`model_state_dict`, `optimizer_state_dict`, `scheduler_state_dict`, `epoch`, `best_val_f1`, `best_epoch`, `epochs_no_improve`) para tolerância a desconexões do Colab.
- **Modo Exploratório Sem Consulta ao Teste (`--val_only`):** Permite comparar durações de janela (30s vs. 60s) ou hiperparâmetros avaliando **exclusivamente** no conjunto de validação (`held_out_test_metrics = null`).
- **Três Variantes Arquiteturais (`--model_variant`):**
  1. `cnn_lstm_gat`: Arquitetura completa `CNN-1D -> LSTM -> GATConv -> FC` com arestas reais entre hosts.
  2. `cnn_lstm_pure`: Baseline temporal puro `CNN-1D -> LSTM -> FC` (`use_gat=False`, sem camada `GATConv`).
  3. `gat_self_loop` (`--ablation_no_graph`): Ablação topológica mantendo `GATConv` apenas com auto-loops $(i, i)$.


## 0. Configuração experimental

Por segurança, o treino completo fica DESABILITADO até que a inspeção de snapshots e os testes sejam concluídos. Use um CSV de captura original enviado à pasta `datasets/` no Drive (`UNSW-NB15.csv` ou `UNSW-NB15.csv.gz`). Não compartilhe chaves, senhas ou credenciais no notebook.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, gzip, hashlib, subprocess, time
import datetime as dt

REPO_URL = "https://github.com/abraaoteixeira/ids-cnn-lstm-gnn.git"
REPO_BRANCH = "fix/leakage-free-pipeline"
WORKDIR = Path("/content/ids-cnn-lstm-gnn")
DRIVE_ROOT = Path("/content/drive/MyDrive/SPECTRE_GRID_NOMS2027")
DATASET_NAME = "UNSW-NB15.csv"   # aceita UNSW-NB15.csv ou UNSW-NB15.csv.gz na pasta datasets/ do Drive
COLAB_SEED = 123                 # reservar seed 42 para Dell G15 (experimentos paralelos independentes)
COLAB_SEEDS = (123, 1337)        # seeds avaliadas no Colab Pro
SELECTED_WINDOW_SECONDS = 30    # decisão pré-registrada, ajustar APENAS antes de inspecionar o teste
MODEL_VARIANT = "cnn_lstm_gat"  # variante principal
COLAB_VARIANTS = ("cnn_lstm_gat", "cnn_lstm_pure", "gat_self_loop", "gat_reverse", "gat_bidir", "gat_residual")
VAL_ONLY_MODE = True            # True durante seleção exploratória/ablações sem consultar o teste reservado
SMOKE_ROWS = 50000
FULL_ACTIVE_ROWS = 200000       # Horizonte ativo do UNSW-NB15_1 (contém 100% dos 22.215 ataques; linhas >186.807 têm 0 ataques)
SMOKE_WINDOWS = (30, 60)
FEATURE_SCOPE = "ebpf_compatible"
RUN_SNAPSHOT_SMOKE = True
RUN_FULL_PREPROCESS = True
RUN_TRAIN = True
TRAIN_EPOCHS = 30
TRAIN_BATCH_SIZE = 8
print("Configuração pronta: FULL_ACTIVE_ROWS=200000, WINDOW=30s, VAL_ONLY_MODE=True, SEEDS=(123, 1337).")


## 1. Verificação da GPU
No Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4 quando disponibilizada). A disponibilidade real é confirmada abaixo.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Habilite GPU no ambiente de execução do Colab antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
total_vram = getattr(props, "total_memory", None)
print("VRAM total (GiB):", round(total_vram / 1024**3, 2) if total_vram else "não informado")
subprocess.run(["nvidia-smi"], check=True)


## 2. Drive persistente
A montagem exige autenticação interativa. Os arquivos do repositório ficam no armazenamento temporário de `/content`, enquanto datasets de origem, snapshots, checkpoints de retomada e relatórios ficam no Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for name in ["datasets", "snapshots", "checkpoints", "reports", "figures", "logs", "manifests"]:
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print("Pasta persistente:", DRIVE_ROOT)


## 3. Código exato da branch de pesquisa
Nenhum checkout ou commit será feito em `main`. O commit efetivamente treinado será registrado no manifesto.


In [ ]:
if not (WORKDIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(WORKDIR)], check=True)
else:
    current = subprocess.check_output(["git", "-C", str(WORKDIR), "branch", "--show-current"], text=True).strip()
    if current != REPO_BRANCH:
        raise RuntimeError("O diretório local está em outra branch: " + current)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
COMMIT_SHA = subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip()
print("Commit congelado nesta sessão:", COMMIT_SHA)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))


## 4. Dependências e teste de GATConv na GPU
Usa o PyTorch CUDA instalado no próprio Colab. Evita substituir uma build CUDA válida por build apenas CPU.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "pandas", "numpy", "scikit-learn",
                "pytest", "matplotlib", "seaborn"], check=True)
import torch_geometric, pandas, sklearn
from torch_geometric.nn import GATConv
conv = GATConv(8, 4, heads=2).cuda()
x = torch.randn(4, 8, device="cuda")
edge_index = torch.tensor([[0, 1, 2], [1, 2, 3]], device="cuda", dtype=torch.long)
with torch.no_grad():
    out = conv(x, edge_index)
torch.cuda.synchronize()
assert out.is_cuda and out.shape == (4, 8)
print("PyG:", torch_geometric.__version__, "| GATConv CUDA OK")


## 5. Gate científico: 9 testes anti-vazamento, causalidade de snapshots e resume
A falha de qualquer teste interrompe o notebook. Salva saída completa no Drive para auditoria.


In [ ]:
pytest_cmd = [sys.executable, "-m", "pytest", "tests/test_leakage_free_pipeline.py", "-v"]
proc = subprocess.run(pytest_cmd, cwd=WORKDIR, text=True, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT)
log_path = DRIVE_ROOT / "logs" / ("colab_pytest_" + COMMIT_SHA[:10] + ".log")
log_path.write_text(proc.stdout, encoding="utf-8")
print(proc.stdout[-16000:])
if proc.returncode != 0:
    raise RuntimeError("Gate científico falhou. Corrigir os testes antes de prosseguir. Log: " + str(log_path))
print("Todos os 9 testes passaram. Log:", log_path)


## 6. Transferência de dados do Dell G15
No G15, envie o `UNSW-NB15.csv` (161 MB) ou `UNSW-NB15.csv.gz` (40 MB) original para `My Drive/SPECTRE_GRID_NOMS2027/datasets/` pelo navegador do Drive ou cliente de sincronização. O CSV nunca deve ser enviado ao repositório GitHub.


In [ ]:
EXPECTED_CSV_BYTES = 168979718
EXPECTED_SHA256 = "7d851bbeabd27894ce39c8e78835c73341fc946652fb7743b9eff193b55eb511"
drive_csv = DRIVE_ROOT / "datasets" / DATASET_NAME
drive_gz = DRIVE_ROOT / "datasets" / (DATASET_NAME + ".gz")
repo_lfs_csv = WORKDIR / "data" / "raw" / "benchmarks" / DATASET_NAME
local_csv = WORKDIR / "data" / "raw" / "colab" / DATASET_NAME
local_csv.parent.mkdir(parents=True, exist_ok=True)

lfs_diag = "não acionado"
if (not drive_csv.is_file() or drive_csv.stat().st_size != EXPECTED_CSV_BYTES) and not drive_gz.is_file():
    if not repo_lfs_csv.is_file() or repo_lfs_csv.stat().st_size != EXPECTED_CSV_BYTES:
        print("[LFS] Verificando instalação do git-lfs e hidratando data/raw/benchmarks/UNSW-NB15.csv...")
        if shutil.which("git-lfs") is None:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "lfs", "install", "--local"], check=True)
        lfs_proc = subprocess.run(
            ["git", "-C", str(WORKDIR), "lfs", "pull", "--include=data/raw/benchmarks/UNSW-NB15.csv"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        lfs_diag = f"returncode={lfs_proc.returncode} | output={lfs_proc.stdout.strip()}"
        print("[LFS]", lfs_diag)
    if repo_lfs_csv.is_file() and repo_lfs_csv.stat().st_size == EXPECTED_CSV_BYTES:
        print("[LFS] Arquivo real hidratado (168.979.718 bytes). Persistindo cópia em Google Drive...")
        shutil.copy2(repo_lfs_csv, drive_csv)

if drive_csv.is_file() and drive_csv.stat().st_size == EXPECTED_CSV_BYTES:
    if not local_csv.exists() or local_csv.stat().st_size != EXPECTED_CSV_BYTES:
        print("Copiando CSV validado do Drive para disco local de execução...")
        shutil.copy2(drive_csv, local_csv)
elif drive_gz.is_file() and drive_gz.stat().st_size > 30000000:
    if not local_csv.exists() or local_csv.stat().st_size != EXPECTED_CSV_BYTES:
        print(f"Descompactando {drive_gz} ({drive_gz.stat().st_size} bytes) para {local_csv}...")
        with gzip.open(drive_gz, "rb") as f_in, open(local_csv, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
elif repo_lfs_csv.is_file() and repo_lfs_csv.stat().st_size == EXPECTED_CSV_BYTES:
    shutil.copy2(repo_lfs_csv, local_csv)

if not local_csv.is_file():
    pointer_preview = ""
    if repo_lfs_csv.is_file():
        pointer_preview = repo_lfs_csv.read_text(encoding="utf-8", errors="replace")[:200]
    raise RuntimeError(
        f"EXECUÇÃO INTERROMPIDA: UNSW-NB15.csv ausente em {local_csv}.\n"
        f"Causa LFS: {lfs_diag}\n"
        f"Conteúdo de {repo_lfs_csv} ({repo_lfs_csv.stat().st_size if repo_lfs_csv.is_file() else 0} bytes): {pointer_preview!r}\n"
        f"Solução: Envie UNSW-NB15.csv.gz (40,79 MB, preparado no Dell G15 em data/processed/UNSW-NB15.csv.gz) para {drive_gz}."
    )

actual_bytes = local_csv.stat().st_size
with open(local_csv, "rb") as f_chk:
    first_line = f_chk.readline().decode("utf-8", errors="replace").strip()
if first_line.startswith("version https://git-lfs.github.com/spec/v1") or actual_bytes != EXPECTED_CSV_BYTES:
    raise RuntimeError(
        f"EXECUÇÃO INTERROMPIDA: {local_csv} está incompleto ou é apenas um ponteiro Git LFS!\n"
        f"Tamanho encontrado: {actual_bytes} bytes (esperado: {EXPECTED_CSV_BYTES} bytes).\n"
        f"Primeira linha: {first_line!r}"
    )

digest = hashlib.sha256()
with open(local_csv, "rb") as f_hash:
    for block in iter(lambda: f_hash.read(4 * 1024 * 1024), b""):
        digest.update(block)
actual_sha256 = digest.hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(f"EXECUÇÃO INTERROMPIDA: SHA-256 divergente ({actual_sha256} != {EXPECTED_SHA256})")

print("Dataset local validado:", local_csv)
print("Tamanho confirmado (bytes):", actual_bytes, f"({round(actual_bytes / (1024**2), 2)} MiB)")
print("SHA-256 confirmado:", actual_sha256)


## 7. Conferência do esquema e da origem temporal
Verificar colunas reais, faixa temporal e prevalência das classes. Um CSV sem origem/destino e horários confiáveis NÃO serve para alegar inferência relacional.


In [ ]:
if not local_csv.is_file() or local_csv.stat().st_size != EXPECTED_CSV_BYTES:
    raise RuntimeError(f"EXECUÇÃO INTERROMPIDA: Dataset inválido ou ausente em {local_csv}")
from preprocessor import load_dataset_csv
sample = load_dataset_csv(str(local_csv), nrows=10000)
print("Linhas de amostra:", len(sample), "| Total de colunas:", len(sample.columns))
print("Colunas:", list(sample.columns))
label_col = "Label" if "Label" in sample.columns else "target"
time_col = "Stime" if "Stime" in sample.columns else ("timestamp" if "timestamp" in sample.columns else None)
src_col = next((c for c in ["srcip", "src_ip", "Source IP", "IPV4_SRC_ADDR"] if c in sample.columns), None)
dst_col = next((c for c in ["dstip", "dst_ip", "Destination IP", "IPV4_DST_ADDR"] if c in sample.columns), None)
if label_col not in sample.columns or time_col is None or src_col is None or dst_col is None:
    raise RuntimeError("Colunas obrigatórias ausentes: label, timestamp, IP origem e IP destino.")
print("Label:", label_col, "| Tempo:", time_col, "| Origem:", src_col, "| Destino:", dst_col)
print("Classes na amostra:", sample[label_col].value_counts(dropna=False).to_dict())
print("Primeiros timestamps:", sample[time_col].iloc[:3].tolist())


## 8. Smoke test de snapshots 30s vs 60s
Os dois bundles são produzidos somente para checagem estrutural, NÃO para selecionar janelas com base nas métricas do teste. Essa amostra de 50 mil linhas é diagnóstica, não serve como evidência final.


In [ ]:
def run_checked(cmd, log_file):
    p = subprocess.run(cmd, cwd=WORKDIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    Path(log_file).write_text(p.stdout, encoding="utf-8")
    print(p.stdout[-4500:])
    if p.returncode:
        raise RuntimeError("Falha ao executar processo; veja o log: " + str(log_file))

if RUN_SNAPSHOT_SMOKE and local_csv.is_file():
    for window in SMOKE_WINDOWS:
        base = "smoke_{}s_{}".format(window, COMMIT_SHA[:10])
        bundle_path = WORKDIR / "data" / "processed" / (base + ".pt")
        features_path = WORKDIR / "data" / "processed" / (base + "_features.json")
        cmd = [sys.executable, "preprocessor.py", "--input_csv", str(local_csv),
               "--window_sec", str(window), "--feature_scope", FEATURE_SCOPE,
               "--max_rows", str(SMOKE_ROWS), "--output_pt", str(bundle_path),
               "--topk_json", str(features_path)]
        run_checked(cmd, DRIVE_ROOT / "logs" / (base + ".log"))
        bundle = torch.load(bundle_path, map_location="cpu", weights_only=False)
        print("Janela:", window, "| protocolo:", bundle.get("protocol"))
        print("Estatísticas:", json.dumps(bundle.get("snapshot_stats", {}), ensure_ascii=False, indent=2))
        assert all(len(bundle[key]) > 0 for key in ("train_snapshots", "val_snapshots", "test_snapshots"))
        shutil.copy2(bundle_path, DRIVE_ROOT / "snapshots" / bundle_path.name)
        shutil.copy2(features_path, DRIVE_ROOT / "snapshots" / features_path.name)
elif RUN_SNAPSHOT_SMOKE:
    print("Smoke test adiado: CSV ausente.")


## 9. Ponto de decisão metodológica

Escolha uma duração de janela usando dados de **TREINO e VALIDAÇÃO** (estatísticas, estabilidade temporal, taxa de classes e F1 de validação). Congele a escolha antes de consultar o TESTE reservado.

Para experimentos exploratórios com várias configurações de janela ou hiperparâmetros, ative `VAL_ONLY_MODE = True` na Célula 0: o `train.py` executará `--val_only`, omitindo completamente a avaliação do conjunto de teste reservado.


## 10. Preparação completa do bundle congelado (desativado por padrão)
Remova o limite de linhas para a execução científica. O bundle fica no Drive para execução reproduzível posterior; o arquivo de features é separado por janela e commit.


In [ ]:
full_bundle_name = "unsw_{}_{}s_{}.pt".format(FEATURE_SCOPE, SELECTED_WINDOW_SECONDS, COMMIT_SHA[:10])
full_bundle_drive = DRIVE_ROOT / "snapshots" / full_bundle_name
full_bundle_local = WORKDIR / "data" / "processed" / full_bundle_name
if RUN_FULL_PREPROCESS:
    if not local_csv.is_file() or local_csv.stat().st_size != EXPECTED_CSV_BYTES:
        raise FileNotFoundError(f"EXECUÇÃO INTERROMPIDA: CSV ausente ou incompleto em {local_csv}")
    features_file = WORKDIR / "data" / "processed" / (full_bundle_name + "_features.json")
    cmd = [sys.executable, "preprocessor.py", "--input_csv", str(local_csv),
           "--window_sec", str(SELECTED_WINDOW_SECONDS),
           "--feature_scope", FEATURE_SCOPE,
           "--output_pt", str(full_bundle_local), "--topk_json", str(features_file)]
    if FULL_ACTIVE_ROWS:
        cmd.extend(["--max_rows", str(FULL_ACTIVE_ROWS)])
    run_checked(cmd, DRIVE_ROOT / "logs" / ("full_preprocess_" + COMMIT_SHA[:10] + ".log"))
    bundle_obj = torch.load(full_bundle_local, map_location="cpu", weights_only=False)
    assert len(bundle_obj["train_snapshots"]) > 0 and len(bundle_obj["val_snapshots"]) > 0 and len(bundle_obj["test_snapshots"]) > 0
    print("Snapshots gerados (200k linhas, 30s, ebpf_compatible):")
    print("  Train snapshots:", len(bundle_obj["train_snapshots"]))
    print("  Val snapshots:  ", len(bundle_obj["val_snapshots"]))
    print("  Test snapshots: ", len(bundle_obj["test_snapshots"]))
    print("  Estatísticas:   ", json.dumps(bundle_obj.get("snapshot_stats", {}), ensure_ascii=False, indent=2))
    shutil.copy2(full_bundle_local, full_bundle_drive)
    shutil.copy2(features_file, DRIVE_ROOT / "snapshots" / features_file.name)
    print("Bundle persistido no Google Drive:", full_bundle_drive, "| bytes:", full_bundle_drive.stat().st_size)
else:
    print("Pré-processamento completo não executado. Altere RUN_FULL_PREPROCESS=True após validação.")


## 11. Treinamento T4 com Retomada Periódica (`--resume`) e Variantes Arquiteturais

Dell G15: `seed 42`; Colab: `seed 123`. Esses treinos são independentes e podem ocorrer simultaneamente. O `train.py` salva a cada época o estado completo (`--resume_state_out` com pesos, otimizador, scheduler, época e melhor Val F1) no Google Drive e retoma automaticamente em caso de desconexão da sessão (`--resume`).

Para comparar ganho topológico real:
- `MODEL_VARIANT = 'cnn_lstm_gat'`: modelo completo com `GATConv` e arestas reais.
- `MODEL_VARIANT = 'cnn_lstm_pure'`: baseline temporal puro `CNN-1D + LSTM` sem camada `GATConv`.
- `MODEL_VARIANT = 'gat_self_loop'`: ablação mantendo `GATConv` apenas com auto-loops.


In [ ]:
run_name = "colab_t4_{}_seed{}_{}s_{}".format(MODEL_VARIANT, COLAB_SEED, SELECTED_WINDOW_SECONDS, COMMIT_SHA[:10])
report_file = DRIVE_ROOT / "reports" / (run_name + ".json")
if RUN_TRAIN:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA indisponível")
    if not full_bundle_drive.is_file():
        raise FileNotFoundError("Bundle completo indisponível: " + str(full_bundle_drive))
    local_train_bundle = WORKDIR / "data" / "processed" / ("training_" + full_bundle_name)
    if not local_train_bundle.exists():
        shutil.copy2(full_bundle_drive, local_train_bundle)
    for seed_val in COLAB_SEEDS:
        for variant_val in COLAB_VARIANTS:
            r_name = "colab_t4_{}_seed{}_{}s_{}".format(variant_val, seed_val, SELECTED_WINDOW_SECONDS, COMMIT_SHA[:10])
            best_ckpt = DRIVE_ROOT / "checkpoints" / (r_name + "_best.pth")
            resume_ckpt = DRIVE_ROOT / "checkpoints" / (r_name + "_resume.pt")
            rep_out = DRIVE_ROOT / "reports" / (r_name + ".json")
            cm_out = DRIVE_ROOT / "figures" / (r_name + "_confusion.png")
            t_log = DRIVE_ROOT / "logs" / (r_name + "_train.log")
            if rep_out.is_file():
                print("[SKIP] Já concluído:", rep_out)
                continue
            print("=== Executando:", r_name, "===")
            cmd = [sys.executable, "train.py", "--data_path", str(local_train_bundle),
                   "--device", "cuda", "--seed", str(seed_val), "--epochs", str(TRAIN_EPOCHS),
                   "--batch_size", str(TRAIN_BATCH_SIZE),
                   "--model_variant", variant_val,
                   "--checkpoint_out", str(best_ckpt),
                   "--resume_state_out", str(resume_ckpt),
                   "--resume", "--save_every", "1",
                   "--report_out", str(rep_out), "--cm_out", str(cm_out)]
            if VAL_ONLY_MODE:
                cmd.append("--val_only")
            run_checked(cmd, t_log)
else:
    print("Treinamento desabilitado. Altere RUN_TRAIN=True SOMENTE após verificação completa.")


## 12. Manifesto e avaliação
O manifesto evita confusão entre dados, commit, duração de janela, feature scope, variante arquitetural e seed. Não divulgar F1 sem evidência do protocolo; não usar o teste para decidir arquitetura.


In [ ]:
manifest = {
    "project": "SPECTRE-GRID NOMS 2027",
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "git_commit": COMMIT_SHA, "git_branch": REPO_BRANCH,
    "dataset_file": DATASET_NAME,
    "selected_window_sec": SELECTED_WINDOW_SECONDS,
    "feature_scope": FEATURE_SCOPE,
    "model_variant": MODEL_VARIANT,
    "val_only_mode": VAL_ONLY_MODE,
    "seed": COLAB_SEED,
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "full_preprocess_requested": RUN_FULL_PREPROCESS,
    "train_requested": RUN_TRAIN,
    "resume_exact_supported": True,
}
if local_csv.is_file():
    digest = hashlib.sha256()
    with open(local_csv, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            digest.update(block)
    manifest["dataset_sha256"] = digest.hexdigest()
manifest_path = DRIVE_ROOT / "manifests" / (run_name + ".json")
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifesto salvo:", manifest_path)
if report_file.is_file():
    report = json.loads(report_file.read_text(encoding="utf-8"))
    print("Métricas de validação:")
    print(json.dumps(report.get("validation_metrics", {}), indent=2, ensure_ascii=False))
    print("Métricas de teste reservado (null se --val_only):")
    print(json.dumps(report.get("held_out_test_metrics", None), indent=2, ensure_ascii=False))
    print("Latência de inferência:")
    print(json.dumps(report.get("inference_latency", None), indent=2, ensure_ascii=False))
else:
    print("Sem relatório final de treino nesta sessão.")


## 13. Critérios científicos e segurança

- Se a GAT elevar o FPR, tratar direção das arestas, heterofilia e ausência de conexão residual como **hipóteses**, a investigar via ablações controladas (`cnn_lstm_gat` vs. `cnn_lstm_pure` vs. `gat_self_loop`). Não declarar causalidade só por diferença de F1.
- Validar que as features ditas `ebpf_compatible` podem ser calculadas no momento exato da inferência real. Uma allowlist por nome não prova paridade eBPF/CSV.
- Os CSVs históricos, pesos originais e a branch `main` NÃO são alterados.
- Sem credenciais no notebook; sem dataset no GitHub; sem acesso automático à rede Tailscale.
- Reprodutibilidade: registrar seed, SHA do commit, checksum do dataset, limiar de decisão e configurações usadas.
